<a href="https://colab.research.google.com/github/RGarancs/applied-ai-academy-labs/blob/main/ccprofit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# C&C Distribution — six markets, four systems

**Applied AI Academy — problem set**

**Used in:** Lesson 8 — AI runs on data plumbing  ·  **Brief:** AIA-C62  
**Rows:** ~22,800 order lines across nine files, 2021 to 2024

---
### Read this before you run anything
This notebook loads the files and stops. It does **not** solve the case, because the
case is not a coding exercise: every tool here will do the arithmetic correctly, and
the arithmetic is not where it goes wrong.

Your job is in the brief. What this gives you is the data in memory and four
questions to answer about it before you compute anything at all.

> Read `cc_distribution.GUIDE.md` first. It is honest and incomplete, and it is
> incomplete in exactly the places that change the answer.


## Setup — load the nine files


In [ ]:
%matplotlib inline
import pandas as pd, numpy as np, os, urllib.request
import matplotlib, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Applied AI Academy chart style — calm, legible when projected
INK, ACCENT, CORE, BUILDER, DANGER = "#26214a", "#544d94", "#5b8a6e", "#a8845c", "#b05a5a"
matplotlib.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c9c4d8", "axes.labelcolor": INK, "axes.titlesize": 13,
    "axes.titleweight": "600", "axes.titlecolor": INK, "axes.titlepad": 14,
    "text.color": INK, "xtick.color": INK, "ytick.color": INK,
    "font.size": 10, "grid.color": "#e6e2ef", "axes.grid": True,
    "axes.axisbelow": True, "grid.linewidth": .8, "figure.facecolor": "white",
})
BASE = "https://appliedai.center/assets/datasets/"
FILES = ["cc_baltics_sales.csv", "cc_poland_sales.csv", "cc_czechia_sales.csv",
         "cc_germany_sales.csv", "cc_returns.csv", "cc_products.csv",
         "cc_promotions.csv", "cc_campaigns.csv", "cc_fx_rates.csv"]
for name in FILES:
    if not os.path.exists(name):
        print("Downloading", name)
        urllib.request.urlretrieve(BASE + name, name)

# Czech order numbers are padded with leading zeros. Read them as text or a
# spreadsheet, and pandas, will eat the padding and the join will silently miss.
B  = pd.read_csv("cc_baltics_sales.csv")
PL = pd.read_csv("cc_poland_sales.csv")
CZ = pd.read_csv("cc_czechia_sales.csv", dtype={"cislo_objednavky": str})
DE = pd.read_csv("cc_germany_sales.csv")
RET   = pd.read_csv("cc_returns.csv", dtype={"order_id": str})
PROD  = pd.read_csv("cc_products.csv")
PROMO = pd.read_csv("cc_promotions.csv")
CAMP  = pd.read_csv("cc_campaigns.csv")
FX    = pd.read_csv("cc_fx_rates.csv")
for n, d in [("Baltics", B), ("Poland", PL), ("Czechia", CZ), ("Germany", DE),
             ("Returns", RET), ("Products", PROD)]:
    print(f"{n:<10} {d.shape}")


---
## Step 1 — look at the four order books side by side

They came off three ERPs and a finance system that have never been reconciled.


In [ ]:
for n, d in [("Baltics", B), ("Poland", PL), ("Czechia", CZ), ("Germany", DE)]:
    print(f"--- {n} ---")
    print(", ".join(d.columns))
    print(d.head(2).to_string(index=False)[:300])
    print()


---
## Step 2 — the four questions, before you compute anything

The cells below do not answer these. They give you what you need to answer them.

1. **What currency is each amount in?** Only one file says.
2. **Is the amount before or after VAT?** The rates differ by market.
3. **What unit is the discount in?** Look at the range in each file.
4. **Does one row mean one thing?** Count rows against distinct ids.


In [ ]:
print("Q3 · the range of every discount column\n")
for n, d, c in [("Baltics", B, "discount"), ("Poland", PL, "rabat_proc"),
                ("Czechia", CZ, "sleva"), ("Germany", DE, "rabatt")]:
    v = pd.to_numeric(d[c], errors="coerce").dropna()
    print(f"{n:<9} {c:<12} min {v.min():>6.2f}   max {v.max():>7.2f}   mean {v.mean():>6.2f}")
print("\nTwo of these are on a different scale from the other two. Which, and what does")
print("that do to an average taken across all four?")


In [ ]:
print("Q4 · one row, one thing?\n")
print(f"returns rows          {len(RET):>7,}")
print(f"distinct return_id    {RET['return_id'].nunique():>7,}")
print(f"difference            {len(RET) - RET['return_id'].nunique():>7,}")
print("\nThe returns log is per ITEM, not per order. Check what a plain merge does to")
print("your row count before you trust anything downstream of it.")


In [ ]:
print("Q1 · the rates you were sent, without comment\n")
print(FX.to_string(index=False))
print("\nNothing in the Polish or Czech order books says which currency they hold.")
print("Q2 · VAT is 19% in Germany, 21-23% elsewhere. One of these four extracts is")
print("gross. The data dictionary says which, in one line, in a note at the end.")


---
## Step 3 — the trap, made visible

Add up the line-value column in each file exactly as delivered, and rank the markets.
Then decide whether you believe the ranking.


In [ ]:
naive = {
    "Czechia": CZ["castka"].sum(),
    "Poland":  pd.to_numeric(PL["wartosc_netto"].astype(str).str.replace(",", "."), errors="coerce").sum(),
    "Baltics": B["net_amount"].sum(),
    "Germany": DE["bruttobetrag"].sum(),
}
print("As the columns stand:")
for k, v in sorted(naive.items(), key=lambda x: -x[1]):
    print(f"   {k:<9} {v:>15,.0f}")
print("\nIs that the real ranking? Answer the four questions above and do it again.")


---
## What you hand in

One page for the board, from the brief:

1. Where the profit comes from, by market, family and channel, net of returns.
2. What C&C should carry and what it should stop.
3. What each discount code costs, and which of them are not marketing at all.
4. Whether the back-to-school campaign pays, with your baseline stated.

Plus a decision log: every assumption you had to supply because the files do not
contain it. That log is the part being marked.
